# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step example for loading and exploring the FAIR² dataset with the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset metadata is described using the [Croissant](https://mlcommons.org/standards/croissant/) schema and accessible via the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load the metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Define the URL to the Croissant metadata
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"\nDataset Description: {metadata.description}")

## 2. Data Overview
Let's review the available record sets, their `@id` fields, and the fields/columns for each RecordSet.
This will help us understand the structure of the data and select subsets for further analysis.


In [ ]:
# List all record sets: @id and name
record_sets = dataset.metadata.recordSet
# recordSet may be None or a singleton if info was not parsed -- fall back
if not record_sets or (isinstance(record_sets, list) and len(record_sets) == 0):
    # Sometimes recordSets are not directly in metadata, but can be inferred by exploring records()
    # We'll attempt to list available RecordSet @id's
    print("No recordSet metadata found directly; inferring record sets from available dataset.records()...")
    # mlcroissant provides a utility
    recordset_ids = dataset.list_record_sets()
    for i, rsid in enumerate(recordset_ids):
        print(f"[{i}] RecordSet @id: {rsid}")
else:
    for i, rs in enumerate(record_sets):
        # Each record set has an @id and probably a name
        print(f"[{i}] RecordSet @id: {rs['@id']} / name: {rs.get('name', 'N/A')}")

print("\n---\nListing fields for each record set:")

recordset_ids = dataset.list_record_sets()
for rsid in recordset_ids:
    print(f"\nRecordSet @id: {rsid}")
    fields = dataset.list_fields(record_set=rsid)
    for field in fields:
        print(f"  Field @id: {field['@id']} (name: {field.get('name', 'N/A')})")

## 3. Data Extraction
Let's select the main record set, extract its records, and load them into a DataFrame for analysis.

**Note:** Use the `@id` of the record set from the previous step.

In [ ]:
# List all available record sets again for reference
recordset_ids = dataset.list_record_sets()
print("Available record set @id's:")
for idx, rsid in enumerate(recordset_ids):
    print(f"  [{idx}] {rsid}")

# For this dataset, typically there will be one main tabular data record set.
# We'll use the first as the main one (update index if needed):
MAIN_RECORDSET_ID = recordset_ids[0]
print(f"\nUsing main record set @id: {MAIN_RECORDSET_ID}")

# Extract records into a DataFrame
records = list(dataset.records(record_set=MAIN_RECORDSET_ID))
df = pd.DataFrame(records)
print("\nColumns (fields' @id):")
print(list(df.columns))
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's apply some basic data processing steps:
- Filtering records based on a numeric field
- Normalizing the numeric field
- Grouping by a categorical field

Use the field `@id`s from the overview above.

In [ ]:
# Display the first few columns for guidance
print(df.columns.tolist())

# Pick a numeric field; in clinical datasets often 'age' is present. We'll select one accordingly.
# Try to find a likely numeric field by scanning column names
numeric_candidates = [c for c in df.columns if 'age' in c.lower() or 'year' in c.lower() or df[c].dtype in [np.int64, np.float64]]
if len(numeric_candidates) == 0:
    # Fallback: try to pick the first numeric-looking column
    for c in df.columns:
        try:
            df[c] = pd.to_numeric(df[c])
            numeric_candidates.append(c)
        except:
            continue

if len(numeric_candidates) == 0:
    raise ValueError("No numeric fields found in this dataset for analysis.")
else:
    print(f"Numeric field candidates: {numeric_candidates}")

# Pick the first candidate for analysis
numeric_field = numeric_candidates[0]
print(f"Using numeric field: {numeric_field}")

# Thresholding: keeping records where the value is greater than a threshold (e.g., age > 50)
try:
    df[numeric_field] = pd.to_numeric(df[numeric_field])
except Exception as e:
    pass

threshold = 50  # e.g., age > 50
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
mean = filtered_df[numeric_field].mean()
std = filtered_df[numeric_field].std()
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean) / std
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Choose a categorical/groupable field (e.g. sex, msi status, cancer_type)
group_candidates = [c for c in df.columns if c != numeric_field and (df[c].dtype == object or 'type' in c.lower() or 'group' in c.lower() or 'status' in c.lower())]

if group_candidates:
    group_field = group_candidates[0]
    print(f"Grouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped (mean {numeric_field}) by {group_field}:")
    print(grouped_df.head())
else:
    print("No appropriate group field found.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field (e.g., Age) and, if available, group it by a categorical variable (e.g., MSI status, Sex).

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df[numeric_field], bins=15, kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If we have a group_field, plot a boxplot
if 'group_field' in locals():
    if group_field in df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f'{numeric_field} by {group_field} (Filtered)')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
In this notebook, we have:
- Loaded clinical data on secondary colorectal cancer survivors from a FAIR²-compliant Croissant dataset using `mlcroissant`.
- Explored available record sets, fields, and columns by referencing their `@id`s.
- Extracted tabular data for analysis and performed basic exploratory data analysis: filtering by a numeric field, normalization, and group comparison.
- Visualized key distributions and observed how grouping variables (such as cancer type or molecular status) relate to key numeric attributes (such as patient age).

For your project, explore additional fields or use machine learning pipelines with the loaded DataFrame and field `@id`s for full reproducibility!